# 01. Data Inspection
This notebook contains memory-safe code to inspect the raw TSV dataset files without loading them entirely into RAM. It outputs schema, row counts, file sizes, and missingness statistics.

In [ ]:
import os
import csv

data_dir = "../../student_resource/dataset"

if not os.path.exists(data_dir):
    data_dir = "../student_resource/dataset"

files_to_check = [
    "train/train_source1.tsv",
    "train/train_source2.tsv",
    "train/train_source3.tsv",
    "train/train_ground_truth.tsv",
    "test/test_source1.tsv",
    "test/test_source2.tsv",
    "test/test_source3.tsv"
]

## File Size & Schema Inspection

In [8]:
for f in files_to_check:
    path = os.path.join(data_dir, f)
    if not os.path.exists(path):
        print(f"Warning: Could not find {path}")
        continue
    size_mb = os.stat(path).st_size / (1024 * 1024)
    print(f"File: {f} ({size_mb:.2f} MB)")
    
    with open(path, 'r', encoding='utf-8') as file:
        header_line = file.readline()
        delimiter = '\t' if '\t' in header_line else ','
        headers = [h.strip() for h in header_line.split(delimiter)]
        print(f"  Headers: {headers}")
        print(f"  Delimiter: {'TAB' if delimiter == '\t' else 'COMMA'}")
        print("  Sample Row:")
        print("  ", file.readline().strip().split(delimiter))
    print("-" * 50)

File: train/train_source1.tsv (200.34 MB)
  Headers: ['entity_id', 'business_name', 'business_address', 'country']
  Delimiter: TAB
  Sample Row:
   ['S1-925783039', "Orelee's Barbershop", '1795 Westchester Drive, High Point, NC', 'US']
--------------------------------------------------
File: train/train_source2.tsv (466.63 MB)
  Headers: ['entity_id', 'business_name', 'business_address', 'country']
  Delimiter: TAB
  Sample Row:
   ['S2-166376419', 'राम मार्केटिंग प्राइवेट लिमिटेड', 'KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi', 'India']
--------------------------------------------------
File: train/train_source3.tsv (480.37 MB)
  Headers: ['entity_id', 'business_name', 'business_address', 'country']
  Delimiter: TAB
  Sample Row:
   ['S3-202863386', 'wilfordhancock.com', 'Mack Rd, Haltom City, Texas', 'US']
--------------------------------------------------
File: train/train_ground_truth.tsv (121.13 MB)
  Headers: ['source1_entity_id', 'matched_entity_ids']
  Delimiter: TAB
  Sample

## Missingness, Uniqueness, and Match Distribution
We stream through the files to compute row counts and missing values without blowing up RAM.

In [9]:
for f in files_to_check:
    path = os.path.join(data_dir, f)
    if not os.path.exists(path): continue
    
    row_count = 0
    duplicate_ids = 0
    unique_ids = set()
    missing = {}
    match_distribution = {}
    
    with open(path, 'r', encoding='utf-8') as file:
        headers = [h.strip() for h in file.readline().split('\t')]
        missing = {h: 0 for h in headers}
        
        for line in file:
            row_count += 1
            cols = [c.strip() for c in line.split('\t')]
            
            # Check missing
            for i, c in enumerate(cols):
                if i < len(headers) and not c:
                    missing[headers[i]] += 1
            
            # Check ID duplicates
            if cols:
                id_val = cols[0]
                if id_val in unique_ids:
                    duplicate_ids += 1
                else:
                    unique_ids.add(id_val)
            
            # Ground truth matches
            if "ground_truth" in f:
                matched = cols[1].split(',') if len(cols)>1 and cols[1] else []
                num_matches = len(matched)
                match_distribution[num_matches] = match_distribution.get(num_matches, 0) + 1
                
    print(f"{f}:")
    print(f"  Total Rows: {row_count:,}")
    print(f"  Duplicate IDs: {duplicate_ids:,}")
    print(f"  Missing Values: {missing}")
    if match_distribution:
        print(f"  Match Distribution (matches -> count):")
        for k, v in sorted(match_distribution.items()):
            print(f"    {k} matches: {v:,} entities")
    print("="*50)

train/train_source1.tsv:
  Total Rows: 2,206,821
  Duplicate IDs: 0
  Missing Values: {'entity_id': 0, 'business_name': 0, 'business_address': 0, 'country': 0}
train/train_source2.tsv:
  Total Rows: 5,034,616
  Duplicate IDs: 0
  Missing Values: {'entity_id': 0, 'business_name': 0, 'business_address': 168967, 'country': 0}
train/train_source3.tsv:
  Total Rows: 5,285,603
  Duplicate IDs: 0
  Missing Values: {'entity_id': 0, 'business_name': 0, 'business_address': 175916, 'country': 0}
train/train_ground_truth.tsv:
  Total Rows: 2,206,821
  Duplicate IDs: 0
  Missing Values: {'source1_entity_id': 0, 'matched_entity_ids': 123247}
  Match Distribution (matches -> count):
    0 matches: 123,247 entities
    1 matches: 119,157 entities
    2 matches: 375,212 entities
    3 matches: 530,841 entities
    4 matches: 484,115 entities
    5 matches: 321,957 entities
    6 matches: 164,868 entities
    7 matches: 63,968 entities
    8 matches: 18,680 entities
    9 matches: 4,205 entities
    10 

## Ground-truth integrity

In [10]:
from collections import Counter

train_dir = os.path.join(data_dir, "train")

s1_path = os.path.join(train_dir, "train_source1.tsv")
s2_path = os.path.join(train_dir, "train_source2.tsv")
s3_path = os.path.join(train_dir, "train_source3.tsv")
gt_path = os.path.join(train_dir, "train_ground_truth.tsv")


def collect_ids(path):
    ids = set()
    with open(path, "r", encoding="utf-8") as f:
        next(f)  # skip header
        for line in f:
            entity_id = line.split("\t", 1)[0].strip()
            ids.add(entity_id)
    return ids


print("Collecting Source 1 IDs...")
s1_ids = collect_ids(s1_path)

print("Collecting Source 2 IDs...")
s2_ids = collect_ids(s2_path)

print("Collecting Source 3 IDs...")
s3_ids = collect_ids(s3_path)

print("\nUnique ID counts:")
print(f"Source 1: {len(s1_ids):,}")
print(f"Source 2: {len(s2_ids):,}")
print(f"Source 3: {len(s3_ids):,}")


missing_s1 = 0
missing_s2 = 0
missing_s3 = 0
malformed_s2 = 0
malformed_s3 = 0
duplicate_matches = 0

gt_rows = 0

with open(gt_path, "r", encoding="utf-8") as f:
    next(f)

    for line in f:
        gt_rows += 1

        source1_id, matched_ids = line.rstrip("\n").split("\t", 1)

        source1_id = source1_id.strip()

        if source1_id not in s1_ids:
            missing_s1 += 1

        if not matched_ids:
            continue

        matches = [x.strip() for x in matched_ids.split(",") if x.strip()]

        # Duplicate IDs inside a single ground-truth match list
        if len(matches) != len(set(matches)):
            duplicate_matches += 1

        for match_id in matches:
            if match_id.startswith("S2-"):
                if match_id not in s2_ids:
                    missing_s2 += 1
            elif match_id.startswith("S3-"):
                if match_id not in s3_ids:
                    missing_s3 += 1
            else:
                if match_id.startswith("S2"):
                    malformed_s2 += 1
                elif match_id.startswith("S3"):
                    malformed_s3 += 1
                else:
                    # Count malformed IDs in either category
                    malformed_s2 += 1

print("\nGROUND-TRUTH INTEGRITY")
print("=" * 60)
print(f"Ground-truth rows: {gt_rows:,}")
print(f"Missing Source-1 IDs: {missing_s1:,}")
print(f"Missing Source-2 referenced IDs: {missing_s2:,}")
print(f"Missing Source-3 referenced IDs: {missing_s3:,}")
print(f"Rows containing duplicate matched IDs: {duplicate_matches:,}")
print(f"Malformed S2/S3 IDs: {malformed_s2 + malformed_s3:,}")


Unique ID counts:
Source 1: 2,206,821
Source 2: 5,034,616
Source 3: 5,285,603

GROUND-TRUTH INTEGRITY
Ground-truth rows: 2,206,821
Missing Source-1 IDs: 0
Missing Source-2 referenced IDs: 0
Missing Source-3 referenced IDs: 0
Rows containing duplicate matched IDs: 0
Malformed S2/S3 IDs: 0


## Matches distribution between Source 2 and Source 3

In [11]:
source2_only = 0
source3_only = 0
both_sources = 0
no_match = 0

total_s2_matches = 0
total_s3_matches = 0

with open(gt_path, "r", encoding="utf-8") as f:
    next(f)

    for line in f:
        _, matched_ids = line.rstrip("\n").split("\t", 1)

        if not matched_ids:
            no_match += 1
            continue

        matches = [x.strip() for x in matched_ids.split(",") if x.strip()]

        has_s2 = any(x.startswith("S2-") for x in matches)
        has_s3 = any(x.startswith("S3-") for x in matches)

        total_s2_matches += sum(x.startswith("S2-") for x in matches)
        total_s3_matches += sum(x.startswith("S3-") for x in matches)

        if has_s2 and has_s3:
            both_sources += 1
        elif has_s2:
            source2_only += 1
        elif has_s3:
            source3_only += 1
        else:
            print("WARNING: Match list contains no S2/S3 IDs")

print("GROUND-TRUTH SOURCE DISTRIBUTION")
print("=" * 60)
print(f"No match:       {no_match:,}")
print(f"Source 2 only:  {source2_only:,}")
print(f"Source 3 only:  {source3_only:,}")
print(f"Both S2 & S3:   {both_sources:,}")
print()
print(f"Total S2 matches: {total_s2_matches:,}")
print(f"Total S3 matches: {total_s3_matches:,}")

GROUND-TRUTH SOURCE DISTRIBUTION
No match:       123,247
Source 2 only:  143,029
Source 3 only:  164,498
Both S2 & S3:   1,776,047

Total S2 matches: 3,693,619
Total S3 matches: 3,944,746


## Country distribution

In [12]:
from collections import Counter

for split in ["train", "test"]:
    print(f"\n{'=' * 60}")
    print(f"{split.upper()} COUNTRY DISTRIBUTIONS")
    print("=" * 60)

    for source in ["source1", "source2", "source3"]:
        path = os.path.join(
            data_dir,
            split,
            f"{split}_{source}.tsv"
        )

        country_counts = Counter()

        with open(path, "r", encoding="utf-8") as f:
            header = next(f).rstrip("\n").split("\t")
            country_idx = header.index("country")

            for line in f:
                cols = line.rstrip("\n").split("\t")

                if len(cols) > country_idx:
                    country = cols[country_idx].strip()
                    country_counts[country] += 1

        total = sum(country_counts.values())

        print(f"\n{source}: {total:,} rows")
        for country, count in country_counts.most_common(20):
            print(f"  {country!r}: {count:,} ({count / total:.2%})")

        if len(country_counts) > 20:
            print(f"  ... {len(country_counts) - 20} additional countries")


TRAIN COUNTRY DISTRIBUTIONS

source1: 2,206,821 rows
  'US': 1,323,633 (59.98%)
  'India': 883,188 (40.02%)

source2: 5,034,616 rows
  'US': 3,016,817 (59.92%)
  'India': 2,017,799 (40.08%)

source3: 5,285,603 rows
  'US': 3,170,056 (59.98%)
  'India': 2,115,547 (40.02%)

TEST COUNTRY DISTRIBUTIONS

source1: 1,732,544 rows
  'India': 809,986 (46.75%)
  'US': 663,106 (38.27%)
  'France': 259,452 (14.98%)

source2: 4,887,273 rows
  'India': 2,312,565 (47.32%)
  'US': 1,871,330 (38.29%)
  'France': 703,378 (14.39%)

source3: 5,082,316 rows
  'India': 2,405,000 (47.32%)
  'US': 1,945,701 (38.28%)
  'France': 731,615 (14.40%)


## Exact duplicate records

In [13]:
import hashlib

for split in ["train", "test"]:
    print(f"\n{'=' * 60}")
    print(f"{split.upper()} EXACT DUPLICATE RECORD CHECK")
    print("=" * 60)

    for source in ["source1", "source2", "source3"]:
        path = os.path.join(
            data_dir,
            split,
            f"{split}_{source}.tsv"
        )

        seen = set()
        duplicate_rows = 0
        total_rows = 0

        with open(path, "rb") as f:
            next(f)  # header

            for line in f:
                total_rows += 1

                # Remove entity_id before hashing so we're checking
                # whether the actual business information is identical.
                parts = line.rstrip(b"\n").split(b"\t", 1)

                if len(parts) != 2:
                    continue

                record_without_id = parts[1]

                digest = hashlib.blake2b(
                    record_without_id,
                    digest_size=16
                ).digest()

                if digest in seen:
                    duplicate_rows += 1
                else:
                    seen.add(digest)

        print(
            f"{source}: "
            f"{duplicate_rows:,} duplicate-content rows "
            f"out of {total_rows:,}"
        )


TRAIN EXACT DUPLICATE RECORD CHECK
source1: 0 duplicate-content rows out of 2,206,821
source2: 25,873 duplicate-content rows out of 5,034,616
source3: 18,860 duplicate-content rows out of 5,285,603

TEST EXACT DUPLICATE RECORD CHECK
source1: 0 duplicate-content rows out of 1,732,544
source2: 22,641 duplicate-content rows out of 4,887,273
source3: 16,293 duplicate-content rows out of 5,082,316


## Field length / missingness characteristics

In [14]:
from collections import Counter

for split in ["train", "test"]:
    print(f"\n{'=' * 60}")
    print(f"{split.upper()} TEXT LENGTH STATISTICS")
    print("=" * 60)

    for source in ["source1", "source2", "source3"]:
        path = os.path.join(
            data_dir,
            split,
            f"{split}_{source}.tsv"
        )

        name_lengths = []
        address_lengths = []

        # Bounded sample: 200,000 rows
        max_rows = 200_000

        with open(path, "r", encoding="utf-8") as f:
            header = next(f).rstrip("\n").split("\t")
            name_idx = header.index("business_name")
            address_idx = header.index("business_address")

            for i, line in enumerate(f):
                if i >= max_rows:
                    break

                cols = line.rstrip("\n").split("\t")

                if len(cols) > address_idx:
                    name_lengths.append(len(cols[name_idx]))
                    address_lengths.append(len(cols[address_idx]))

        name_lengths.sort()
        address_lengths.sort()

        def percentile(values, p):
            if not values:
                return 0
            idx = int((len(values) - 1) * p)
            return values[idx]

        print(f"\n{source} (sample: {len(name_lengths):,})")

        print(
            "  Name length: "
            f"median={percentile(name_lengths, .50)}, "
            f"p90={percentile(name_lengths, .90)}, "
            f"p99={percentile(name_lengths, .99)}, "
            f"max={max(name_lengths)}"
        )

        print(
            "  Address length: "
            f"median={percentile(address_lengths, .50)}, "
            f"p90={percentile(address_lengths, .90)}, "
            f"p99={percentile(address_lengths, .99)}, "
            f"max={max(address_lengths)}"
        )


TRAIN TEXT LENGTH STATISTICS

source1 (sample: 200,000)
  Name length: median=24, p90=34, p99=42, max=81
  Address length: median=41, p90=90, p99=124, max=222

source2 (sample: 200,000)
  Name length: median=25, p90=37, p99=48, max=104
  Address length: median=37, p90=83, p99=118, max=204

source3 (sample: 200,000)
  Name length: median=25, p90=38, p99=50, max=81
  Address length: median=42, p90=78, p99=115, max=200

TEST TEXT LENGTH STATISTICS

source1 (sample: 200,000)
  Name length: median=24, p90=34, p99=42, max=92
  Address length: median=50, p90=93, p99=125, max=240

source2 (sample: 200,000)
  Name length: median=25, p90=38, p99=49, max=87
  Address length: median=43, p90=87, p99=120, max=226

source3 (sample: 200,000)
  Name length: median=25, p90=38, p99=50, max=82
  Address length: median=43, p90=81, p99=117, max=235


In [ ]:
import random
import re
import unicodedata
from collections import Counter

# Conservative normalization for initial analysis only
def normalize_text(text):
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)

    return text


# Load only the records needed for sampled GT pairs
random.seed(42)

SAMPLE_PAIRS = 20_000

# First collect a random sample of positive pairs from GT
sampled_pairs = []

with open(gt_path, "r", encoding="utf-8") as f:
    next(f)

    for line in f:
        source1_id, matched_ids = line.rstrip("\n").split("\t", 1)

        if not matched_ids:
            continue

        matches = [x.strip() for x in matched_ids.split(",") if x.strip()]

        for match_id in matches:
            # Reservoir sampling
            pair = (source1_id, match_id)

            if len(sampled_pairs) < SAMPLE_PAIRS:
                sampled_pairs.append(pair)
            else:
                j = random.randint(0, len(sampled_pairs))
                if j < SAMPLE_PAIRS:
                    sampled_pairs[j] = pair


print(f"Sampled positive pairs: {len(sampled_pairs):,}")


# Extract IDs required from the sample
needed_s1 = set()
needed_s2 = set()
needed_s3 = set()

for s1_id, match_id in sampled_pairs:
    needed_s1.add(s1_id)

    if match_id.startswith("S2-"):
        needed_s2.add(match_id)
    elif match_id.startswith("S3-"):
        needed_s3.add(match_id)


def load_required_records(path, required_ids):
    records = {}

    with open(path, "r", encoding="utf-8") as f:
        header = next(f).rstrip("\n").split("\t")
        id_idx = header.index("entity_id")
        name_idx = header.index("business_name")
        address_idx = header.index("business_address")
        country_idx = header.index("country")

        for line in f:
            cols = line.rstrip("\n").split("\t")

            if len(cols) <= country_idx:
                continue

            entity_id = cols[id_idx]

            if entity_id in required_ids:
                records[entity_id] = {
                    "name": cols[name_idx],
                    "address": cols[address_idx],
                    "country": cols[country_idx]
                }

    return records


print("Loading sampled Source-1 records...")
s1_records = load_required_records(s1_path, needed_s1)

print("Loading sampled Source-2 records...")
s2_records = load_required_records(s2_path, needed_s2)

print("Loading sampled Source-3 records...")
s3_records = load_required_records(s3_path, needed_s3)


# Compare sampled positive pairs
stats = {
    "S2": {
        "pairs": 0,
        "name_exact": 0,
        "name_norm_exact": 0,
        "address_exact": 0,
        "address_norm_exact": 0,
        "country_exact": 0,
    },
    "S3": {
        "pairs": 0,
        "name_exact": 0,
        "name_norm_exact": 0,
        "address_exact": 0,
        "address_norm_exact": 0,
        "country_exact": 0,
    }
}


for s1_id, match_id in sampled_pairs:

    if s1_id not in s1_records:
        continue

    if match_id.startswith("S2-"):
        source = "S2"
        target_records = s2_records
    else:
        source = "S3"
        target_records = s3_records

    if match_id not in target_records:
        continue

    a = s1_records[s1_id]
    b = target_records[match_id]

    stats[source]["pairs"] += 1

    if a["name"] == b["name"]:
        stats[source]["name_exact"] += 1

    if normalize_text(a["name"]) == normalize_text(b["name"]):
        stats[source]["name_norm_exact"] += 1

    if a["address"] == b["address"]:
        stats[source]["address_exact"] += 1

    if normalize_text(a["address"]) == normalize_text(b["address"]):
        stats[source]["address_norm_exact"] += 1

    if a["country"] == b["country"]:
        stats[source]["country_exact"] += 1


print("\nPOSITIVE-PAIR CHARACTERISTICS")
print("=" * 70)

for source, s in stats.items():

    n = s["pairs"]

    print(f"\n{source} positive pairs: {n:,}")

    if n == 0:
        continue

    print(
        f"  Name exact:             {s['name_exact'] / n:.2%}"
    )

    print(
        f"  Name normalized exact:  {s['name_norm_exact'] / n:.2%}"
    )

    print(
        f"  Address exact:          {s['address_exact'] / n:.2%}"
    )

    print(
        f"  Address normalized:     {s['address_norm_exact'] / n:.2%}"
    )

    print(
        f"  Country exact:          {s['country_exact'] / n:.2%}"
    )

Sampled positive pairs: 20,000
Loading sampled Source-1 records...
Loading sampled Source-2 records...
Loading sampled Source-3 records...

POSITIVE-PAIR CHARACTERISTICS

S2 positive pairs: 9,705
  Name exact:             4.74%
  Name normalized exact:  18.41%
  Address exact:          0.00%
  Address normalized:     12.09%
  Country exact:          100.00%

S3 positive pairs: 10,295
  Name exact:             4.20%
  Name normalized exact:  18.28%
  Address exact:          4.30%
  Address normalized:     4.31%
  Country exact:          100.00%
